# Classwork: The Analyst's Toolkit

**Names:** _(pair)_

After learning about different tools (or a toolkit-- Descriptive statistics, Correlation, Regression/Multiple regression, Chi-square test) of working with data in the previous Mods, in this acitivity you will apply your real skills on the data you have.  
  
  Suppose your company hands you `customers.csv` (40 customers: `age`, `income`, `spending`, `region`, `subscribed`) and a list of questions. You are the analyst. Answer each one with the right tool/technique you decide to apply.

Work in pairs. Write answers in the markdown cells provided.

## Setup
Import the libraries you'll need and load the data.

In [17]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm

df = pd.read_csv("./customers.csv")
print(df.head(20))
df.info()


    customer_id  age  income  spending region subscribed
0             1   25    61.3      39.1   West         No
1             2   25    54.9      42.1  South        Yes
2             3   51   105.1      59.9  South         No
3             4   39     NaN      49.9  South         No
4             5   43    72.4      48.3   East         No
5             6   44    84.7      47.1   East         No
6             7   48    92.9      60.0   East        Yes
7             8   21    53.3      36.0  South         No
8             9   39    71.9      48.9   East        Yes
9            10   25    70.4      42.1   east        Yes
10           11   36    92.2      62.6   East        Yes
11           12   57   106.1      66.7   East        Yes
12           13   41    81.8      49.8  South        Yes
13           14   22    70.2      48.9   West         No
14           15   41    96.4      65.8   East        Yes
15           16   25    67.0      54.2   West        Yes
16           17   50   105.4   

## Part 0: Know your data, then clean it

This data is dirty on purpose. Find and fix **three** problems before you analyze anything.

In [18]:
print(df.isna().sum())
print(df['region'].unique())
print(df['income'].describe())

df['region'] = df['region'].str.title()

df.loc[df['income'] < 0, 'income'] = np.nan

df['income'] = df['income'].fillna(df['income'].median())

print(df.isna().sum())
print(df['region'].unique())
print(df['income'].min())

customer_id    0
age            0
income         2
spending       0
region         0
subscribed     0
dtype: int64
<StringArray>
['West', 'South', 'East', 'east']
Length: 4, dtype: str
count     38.000000
mean      83.007895
std       27.628563
min      -50.000000
25%       72.975000
50%       84.700000
75%       98.500000
max      114.500000
Name: income, dtype: float64
customer_id    0
age            0
income         0
spending       0
region         0
subscribed     0
dtype: int64
<StringArray>
['West', 'South', 'East']
Length: 3, dtype: str
53.3


**Explain each issue:** what you found and how you handled it.

Income had two missing values so I filled them with the median because if we did the mean, it would be affected by the outliers. There was also bad capitalization in region, like east and East that was counting as different things. So str.title() made it so east -> East and it works correctly now. There is also  section in income that is -50, which is impossible. So we did a <0 to make sure nothing can be below 0, and anything below zero is replaced with NaN. Then we replaced the NaN with the median (using fillna) like we did earlier.


## Part 1: Pick the tool for each question

For each question: (a) name the data types involved, (b) name the tool, (c) run it, (d) answer in one plain sentence.

### Q1. Do customers who earn more also spend more?

In [19]:
r, p = stats.pearsonr(df['income'], df['spending'])
print('r =', round(r, 3))
print('p-value =', p)

r = 0.717
p-value = 1.9185294399175956e-07


*Your answer:* There isa strong positive correlation between income and spending because r is less than 1 and p is less than 0.05


### Q2. Is a customer's region related to whether they subscribed?

In [ ]:
table = pd.crosstab(df['region'], df['subscribed'])
print(table)

chi2, p, dof, expected = stats.chi2_contingency(table)
print('chi2 =', round(chi2, 2))
print('p-value =', round(p, 4))
print('dof =', dof)

subscribed  No  Yes
region             
East         5   16
South        6    3
West         9    1
chi2 = 13.16
p-value = 0.0014
dof = 2


: 

*Your answer:* Yes they are related because the chi test is p is less than 0.05. It seems like the East customers subscribed while majority of the West customers didn't.


### Q3. Can we predict spending from income and age together? How much do they explain? (You can review your Assignment 4 for this)

In [ ]:
X = df[['income', 'age']]
X = sm.add_constant(X)
y = df['spending']

model = sm.OLS(y, X).fit()
print(model.summary())


*Your answer:* Yes we can preditct spending from income and age. They explain 52% of variation in spending because r^2 = 0.52, and income is significant while age is not.


### Q4. Which region has the most customers, and what is the average income overall?

In [ ]:
print(df['region'].value_counts())
print('Average income:', round(df['income'].mean(), 2))

*Your answer:* The East region has the most customers. The average income overall is about 86.46.


## Part 2: The reveal

Answer these two in a sentence each (no code needed).

**a)** Look back at your four tools (Correlation, Chi-square test, Multiple regression, & Descriptive statistic)). What decided which tool you used each time? (Hint: it was not the wording of the question.)

*Your answer:* It was the data types that decided which tool we used.


**b)** In Q1 you found a relationship. Can you tell the company that higher income *causes* higher spending? Why or why not?

*Your answer:* No because correlation does not mean causation. Something ouside of income and spending could be affecting it.


## Bonus

In Q2 you ran a chi-square test. Look at its **expected counts** (not the observed counts). Do they all meet the assumption you learned in the chi-square module? What would you tell the company about how much to trust that result?

In [ ]:
print(pd.DataFrame(expected, index=table.index, columns=table.columns))
print('All expected counts >= 5?', (expected >= 5).all())


*Your answer:* No because he rule is that every expected count should be at least 5, but the South row has expected counts of 4.5. Because of that I wouldn't trust the results fully because there is not enough data.


### Push your code to GitHub